In [1]:
import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

In [2]:
data_path = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/RFSOC_v3_1020_reoder_v3_FesSta_FD2_withoutNewCattle_reorganize_1114final.csv"       
out_dir   = "./dsm_cluster_vif_outputs"

In [3]:
feature_slice = ("PS_b1_point", "vpdmax_fourier2_phase_rad")

In [10]:
keep_first_cols = 0  
keep_last_cols  = 0   

In [4]:
linkage_method = "average" 
cluster_dist_threshold = 0.2 

vif_threshold = 10.0          
min_keep_per_cluster = 1      
max_iter_per_cluster = 1000  

In [5]:
varinfo_path = None

In [6]:
fig_dpi = 250

In [7]:
os.makedirs(out_dir, exist_ok=True)

In [8]:
df_raw = pd.read_csv(data_path)
df = df_raw.copy()

In [11]:
first_cols = df.columns[:keep_first_cols]
last_cols  = df.columns[-keep_last_cols:] if keep_last_cols > 0 else pd.Index([])

if feature_slice is not None:
    start_col, end_col = feature_slice
    A = df.loc[:, start_col:end_col].copy()
else:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    excluded = set(first_cols).union(set(last_cols))
    feat_cols = [c for c in numeric_cols if c not in excluded]
    A = df.loc[:, feat_cols].copy()

In [12]:
print(f"[INFO] 原始特征数: {A.shape[1]}, 样本数: {A.shape[0]}")

[INFO] 原始特征数: 363, 样本数: 1019


In [13]:
nunique = A.nunique(dropna=False)
zero_var_cols = nunique[nunique <= 1].index.tolist()
if zero_var_cols:
    print(f"[CLEAN] 删除零方差列: {len(zero_var_cols)}")
    A.drop(columns=zero_var_cols, inplace=True)

In [14]:
dup_mask = A.T.duplicated(keep="first")

dup_cols = A.columns[dup_mask]

print("[CLEAN] 重复列有:")
print(dup_cols.tolist())

A_T = A.T.drop_duplicates(keep="first")
A = A_T.T

[CLEAN] 重复列有:
['gypsum_15_cm_p']


In [15]:
A_T = A.T.drop_duplicates(keep="first")
if A_T.shape[0] < A.shape[1]:
    dup_cnt = A.shape[1] - A_T.shape[0]
    print(f"[CLEAN] 删除重复列: {dup_cnt}")
    A = A_T.T

In [16]:
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
import numpy as np
import pandas as pd

is_const = A.nunique(dropna=True) <= 1
if is_const.any():
    print(f"[WARN] 去除常数列: {A.columns[is_const].tolist()}")
    A = A.loc[:, ~is_const].copy()

corr = A.corr(method="pearson")

corr = corr.clip(-1.0, 1.0)              
corr = corr.fillna(0.0)                  

dist_mat = 1.0 - np.abs(corr.values)
np.fill_diagonal(dist_mat, 0.0)

dist_mat = 0.5 * (dist_mat + dist_mat.T)

dist_mat = np.nan_to_num(dist_mat, nan=0.0, posinf=1.0, neginf=0.0)
dist_mat = np.clip(dist_mat, 0.0, 1.0)

min_d, max_d = dist_mat.min(), dist_mat.max()
if min_d < 0:
    print(f"[WARN] 距离最小值 {min_d:.3e} < 0，已截断为 0。")
if not np.allclose(dist_mat, dist_mat.T, atol=1e-12):
    raise ValueError("距离矩阵不是对称的。")
if np.any(np.diag(dist_mat) != 0.0):
    raise ValueError("距离矩阵对角不为 0。")

dist_vec = squareform(dist_mat, checks=True)

if linkage_method.lower() == "ward":
    raise ValueError("method='ward' 不能用于预计算距离。请改用 'average' / 'complete'，"
                     "或传观测矩阵且 metric='euclidean'。")

linked = linkage(dist_vec, method=linkage_method)

clusters = fcluster(linked, t=cluster_dist_threshold, criterion="distance")
cluster_df = pd.DataFrame({"Var": A.columns, "Cluster": clusters})
cluster_df_sorted = cluster_df.sort_values("Cluster").reset_index(drop=True)

cluster_df_sorted.to_csv(os.path.join(out_dir, "cluster_assignments_sorted.csv"), index=False)
cluster_df.to_csv(os.path.join(out_dir, "cluster_assignments.csv"), index=False)
pd.DataFrame(corr, index=A.columns, columns=A.columns).to_csv(os.path.join(out_dir, "corr_matrix.csv"))

print(f"[INFO] 簇数量: {cluster_df['Cluster'].nunique()}")

[INFO] 簇数量: 150


In [17]:
def plot_clustermap(corr_matrix, cluster_meta=None, save_path=None):

    sns.set_context("notebook")
    plt.figure(figsize=(6, 6))  
    plt.close()

    col_colors = None
    legend_handles = []

    if cluster_meta is not None and {"Factor", "Subcategory"}.issubset(cluster_meta.columns):

        factors = cluster_meta["Factor"].fillna("NA")
        subcats = cluster_meta["Subcategory"].fillna("NA")

        factor_unique = factors.unique()
        subcat_unique = subcats.unique()

        factor_palette = sns.color_palette("ch:s=.25,rot=-.5", n_colors=len(factor_unique))
        subcat_palette = sns.color_palette("Paired", n_colors=len(subcat_unique))

        factor_map = dict(zip(factor_unique, factor_palette))
        subcat_map = dict(zip(subcat_unique, subcat_palette))

        col_colors = pd.DataFrame({
            "Factor": factors.map(factor_map),
            "Subcategory": subcats.map(subcat_map)
        }, index=cluster_meta.index)

        for k, c in factor_map.items():
            legend_handles.append(Patch(facecolor=c, label=f"Factor: {k}"))
        for k, c in subcat_map.items():
            legend_handles.append(Patch(facecolor=c, label=f"Subcategory: {k}"))

        g = sns.clustermap(
            corr_matrix, cmap="coolwarm", center=0,
            col_colors=col_colors, row_colors=col_colors,
            xticklabels=False, yticklabels=False
        )

        g.ax_col_dendrogram.legend(
            handles=legend_handles, title="Categories",
            loc="center left", bbox_to_anchor=(1.05, 0.5), borderaxespad=0.
        )
    else:
        g = sns.clustermap(
            corr_matrix, cmap="coolwarm", center=0,
            xticklabels=False, yticklabels=False
        )

    if save_path:
        plt.savefig(save_path, dpi=fig_dpi, bbox_inches="tight")
    plt.close()

In [18]:
meta_for_colors = None
if varinfo_path and os.path.exists(varinfo_path):
    varinfo = pd.read_csv(varinfo_path)
    # 需要含列: Var, Factor, Subcategory
    needed = {"Var", "Factor", "Subcategory"}
    if needed.issubset(varinfo.columns):
        meta_for_colors = varinfo.set_index("Var").reindex(A.columns)
    else:
        print("[WARN] varinfo 缺少列，跳过着色：需要列 {Var, Factor, Subcategory}")

In [19]:
plot_clustermap(
    corr_matrix=corr,
    cluster_meta=meta_for_colors,
    save_path=os.path.join(out_dir, "clustermap.png")
)
print("[INFO] clustermap 已保存。")

[INFO] clustermap 已保存。


In [20]:
def calculate_vif_df(X_df: pd.DataFrame) -> pd.DataFrame:
    X = X_df.values
    cols = list(X_df.columns)
    vif_vals = []
    for i in range(X.shape[1]):
        try:
            v = variance_inflation_factor(X, i)
        except Exception:
            v = np.inf
        vif_vals.append(v)
    return pd.DataFrame({"Var": cols, "VIF": vif_vals})

In [21]:
def reduce_by_vif_within_clusters(A_df: pd.DataFrame,
                                  cluster_map: pd.DataFrame,
                                  vif_thr: float = 10.0,
                                  min_keep: int = 1,
                                  max_iter: int = 1000):
    status_rows = []
    kept_vars = []

    status = pd.DataFrame({
        "Var": cluster_map["Var"].values,
        "Cluster": cluster_map["Cluster"].values,
        "status_vif": "keep",
        "last_vif": np.nan,
        "removed_at_iter": np.nan,
    })

    for c in sorted(cluster_map["Cluster"].unique()):
        vars_in_c = cluster_map[cluster_map["Cluster"] == c]["Var"].tolist()
        subA = A_df.loc[:, vars_in_c].copy()
        it = 0

        while subA.shape[1] > min_keep and it < max_iter:
            it += 1
            vif_df = calculate_vif_df(subA)
            max_vif = vif_df["VIF"].max()

            if max_vif < vif_thr:

                for _, r in vif_df.iterrows():
                    status.loc[status["Var"] == r["Var"], "last_vif"] = r["VIF"]
                break

            worst = vif_df.loc[vif_df["VIF"].idxmax(), "Var"]
            status.loc[status["Var"] == worst, ["status_vif", "last_vif", "removed_at_iter"]] = ["delete", max_vif, it]
            subA = subA.drop(columns=[worst])

        final_vif = calculate_vif_df(subA)
        for _, r in final_vif.iterrows():
            status.loc[status["Var"] == r["Var"], "last_vif"] = r["VIF"]

        kept_vars.extend(list(subA.columns))

    status = status.merge(cluster_map, on=["Var", "Cluster"], how="left")
    status = status[["Var", "Cluster", "status_vif", "last_vif", "removed_at_iter"]].sort_values(
        by=["Cluster", "status_vif", "last_vif"],
        ascending=[True, True, True]
    )
    return status, kept_vars

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
A_std = pd.DataFrame(
    scaler.fit_transform(A), 
    columns=A.columns, 
    index=A.index
)


In [23]:
status_df, kept_vars = reduce_by_vif_within_clusters(
    A_df=A_std,  
    cluster_map=cluster_df,
    vif_thr=vif_threshold,
    min_keep=min_keep_per_cluster,
    max_iter=max_iter_per_cluster
)

In [24]:
status_df.to_csv(os.path.join(out_dir, "vif_reduction_log.csv"), index=False)
pd.Series(kept_vars, name="kept_vars").to_csv(os.path.join(out_dir, "kept_vars.txt"), index=False)
print(f"[INFO] VIF后保留变量数: {len(kept_vars)}")

[INFO] VIF后保留变量数: 211


In [1]:
import os
import pandas as pd

out_dir = "./dsm_cluster_vif_outputs"
os.makedirs(out_dir, exist_ok=True)

corr_path = os.path.join(out_dir, "corr_matrix.csv")
corr = pd.read_csv(corr_path, index_col=0)

tmpl = pd.DataFrame({
    "Var": corr.columns.tolist(),
    "Factor": [""] * corr.shape[1],
    "Subcategory": [""] * corr.shape[1],
})

tmpl_path = os.path.join(out_dir, "varinfo_template.csv")
tmpl.to_csv(tmpl_path, index=False, encoding="utf-8-sig")
print("[OK] 已生成模板：", tmpl_path)
print("      请你打开 varinfo_template.csv，填好 Factor 与 Subcategory 两列后保存。")


[OK] 已生成模板： ./dsm_cluster_vif_outputs/varinfo_template.csv
      请你打开 varinfo_template.csv，填好 Factor 与 Subcategory 两列后保存。


In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

out_dir = "./dsm_cluster_vif_outputs"
corr_path = os.path.join(out_dir, "corr_matrix.csv")
varinfo_path = os.path.join(out_dir, "varinfo_template.csv")  
fig_dpi = 250

corr = pd.read_csv(corr_path, index_col=0)
corr = corr.loc[corr.columns, corr.columns] 

varinfo = pd.read_csv(varinfo_path)
need = {"Var", "Factor", "Subcategory"}
if not need.issubset(varinfo.columns):
    raise ValueError(f"varinfo 缺少列：{need}，你当前列为：{varinfo.columns.tolist()}")

meta = varinfo.set_index("Var").reindex(corr.columns)
meta["Factor"] = meta["Factor"].fillna("NA").replace("", "NA")
meta["Subcategory"] = meta["Subcategory"].fillna("NA").replace("", "NA")

factors = meta["Factor"].astype(str)
subcats  = meta["Subcategory"].astype(str)

factor_unique = factors.unique().tolist()
subcat_unique = subcats.unique().tolist()

factor_palette = sns.color_palette("tab20", n_colors=len(factor_unique))
subcat_palette = sns.color_palette("husl",  n_colors=len(subcat_unique))

factor_map = dict(zip(factor_unique, factor_palette))
subcat_map = dict(zip(subcat_unique, subcat_palette))

col_colors = pd.DataFrame({
    "Factor": factors.map(factor_map),
    "Subcategory": subcats.map(subcat_map)
}, index=corr.columns)

g = sns.clustermap(
    corr,
    cmap="coolwarm",
    center=0,
    col_colors=col_colors,
    row_colors=col_colors,
    xticklabels=False,
    yticklabels=False
)

reordered = g.dendrogram_col.reordered_ind
ordered_vars = corr.columns[reordered]

col_colors_reordered = col_colors.loc[ordered_vars]

clustermap_out = os.path.join(out_dir, "clustermap_with_bars.png")
plt.savefig(clustermap_out, dpi=fig_dpi, bbox_inches="tight")
plt.close()
print("[OK] clustermap（自带bar）已保存：", clustermap_out)

def save_colorbar_png(color_list, save_path, height_px=60):

    N = len(color_list)
    arr = np.array(color_list, dtype=float).reshape(1, N, 3)

    dpi = fig_dpi
    height_in = height_px / dpi
    width_in = max(6, N / 40)  # 经验值：N越多图越宽

    fig, ax = plt.subplots(figsize=(width_in, height_in), dpi=dpi)
    ax.imshow(arr, aspect="auto")
    ax.set_axis_off()
    plt.savefig(save_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close()

factor_bar_path = os.path.join(out_dir, "factor_colorbar_reordered.png")
subcat_bar_path = os.path.join(out_dir, "subcategory_colorbar_reordered.png")

save_colorbar_png(
    color_list=col_colors_reordered["Factor"].tolist(),
    save_path=factor_bar_path,
    height_px=60
)
save_colorbar_png(
    color_list=col_colors_reordered["Subcategory"].tolist(),
    save_path=subcat_bar_path,
    height_px=60
)

print("[OK] Factor bar 已保存：", factor_bar_path)
print("[OK] Subcategory bar 已保存：", subcat_bar_path)

legend_handles = []
for k, c in factor_map.items():
    legend_handles.append(Patch(facecolor=c, edgecolor="none", label=f"Factor: {k}"))
for k, c in subcat_map.items():
    legend_handles.append(Patch(facecolor=c, edgecolor="none", label=f"Subcategory: {k}"))

legend_path = os.path.join(out_dir, "colorbar_legend.png")
fig = plt.figure(figsize=(8, max(3, len(legend_handles) * 0.18)))
plt.axis("off")
plt.legend(handles=legend_handles, loc="center left", frameon=False)
plt.savefig(legend_path, dpi=fig_dpi, bbox_inches="tight")
plt.close()
print("[OK] 图例已保存：", legend_path)

order_out = os.path.join(out_dir, "ordered_vars_for_heatmap.csv")
pd.DataFrame({"Var_reordered": ordered_vars}).to_csv(order_out, index=False, encoding="utf-8-sig")
print("[OK] heatmap最终横向顺序已保存：", order_out)


/apps/geopython/1.0.1/lib/python3.7/site-packages/seaborn/matrix.py:649: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)


[OK] clustermap（自带bar）已保存： ./dsm_cluster_vif_outputs/clustermap_with_bars.png
[OK] Factor bar 已保存： ./dsm_cluster_vif_outputs/factor_colorbar_reordered.png
[OK] Subcategory bar 已保存： ./dsm_cluster_vif_outputs/subcategory_colorbar_reordered.png
[OK] 图例已保存： ./dsm_cluster_vif_outputs/colorbar_legend.png
[OK] heatmap最终横向顺序已保存： ./dsm_cluster_vif_outputs/ordered_vars_for_heatmap.csv
